# Volume 7, Chapter 7: Fourier Analysis on Finite Abelian Groups (Python companion)

Exercises first, solutions at the end. A fast Fourier transform for every length, Chebotarev's lemma and sparse recovery, caps in the game of SET, and the linearity test of Blum, Luby and Rubinfeld. Requirements: numpy.

*Before each solution, predict what the output will look like.*

## Exercises

**P7.1 (★★) CS application: a fast transform for every length.** Implement Lemma 7.7 recursively for $G=\mathbb Z/n\mathbb Z$, splitting along the subgroup of index $p$ for the smallest prime $p\mid n$, and count the operations. Check it against the definition and against `numpy.fft` for $n=2^{10}$, $3^6$, $1000$ and $997$, and compare the counts with the bound of Theorem 7.8. Then write the Walsh–Hadamard transform of $\mathbb F_2^n$ in place, in $n2^n$ additions.

**P7.2 (★★) Chebotarev, Tao and sparse recovery.** Check Lemma 7.10 for $p=7$ by computing all $3431$ square minors of the DFT matrix and their smallest absolute value; do the same for $n=8$ and count the singular minors. Then take a random $3$-sparse signal on $\mathbb Z/31\mathbb Z$, keep $6$ randomly chosen Fourier coefficients, and recover the signal by trying all supports of size $3$. Is the answer unique, as Theorem 7.11 promises? What happens on $\mathbb Z/32\mathbb Z$ with a signal supported on a subgroup?

**P7.3 (★) Playing SET.** Build caps in $\mathbb F_3^n$ for $n=1,\dots,6$ by a randomised greedy algorithm (add random points while no three sum to $0$, keep the best of many runs), and compare with the true maxima $2,4,9,20,45,112$, with Meshulam's bound $\frac{2\cdot3^n}n$ and with $2.756^n$. Verify the cap property with the Fourier count of Lemma 7.15, computed by a multidimensional FFT. In a random deal of $12$ SET cards, how often is there no set?

**P7.4 (★) The BLR test in action.** For $n=10$, estimate by simulation the acceptance probability of the BLR test for: a linear function; a linear function with $5\%$ and $20\%$ of its values flipped; the majority of the first three bits; a uniformly random function. Compare with the exact formula $\frac12+\frac12\sum_S\hat F(S)^3$ of Theorem 7.17, computed with your Walsh–Hadamard transform, and with the distance to the nearest linear function.

## Solutions

### P7.1

In [ ]:
import numpy as np, itertools, math, time

ops = 0
def smallest_prime(n):
    p = 2
    while p * p <= n:
        if n % p == 0:
            return p
        p += 1
    return n

def fft(f):
    """Fourier transform hat f(b) = E_x f(x) e(-bx/n) on Z/n, by Lemma 7.7 with H = pZ/nZ."""
    global ops
    n = len(f)
    if n == 1:
        return f.copy()
    p = smallest_prime(n)
    if p == n:                                   # prime length: from the definition
        ops += n * n
        x = np.arange(n)
        return np.exp(-2j * np.pi * np.outer(x, x) / n) @ f / n
    m = n // p                                   # H = pZ/nZ = {p*h}, |H| = m; T = {0,...,p-1}
    F = np.array([fft(f[t::p]) for t in range(p)])   # F[t, psi] = transform on H of h -> f(t + p h)
    # characters of H: psi_c(p h) = e(c h / m); extension chi_psi(x) = e(c x / n); eta in H^perp: e(j m x / n)
    c = np.arange(m)
    tw = np.exp(-2j * np.pi * np.outer(np.arange(p), c) / n)     # conj(chi_psi(t))
    ops += n
    G = (tw * F) / p
    out = np.empty(n, dtype=complex)
    w = np.exp(-2j * np.pi * np.outer(np.arange(p), np.arange(p)) / p)
    ops += m * p * p
    for ci in range(m):                          # for each psi, a transform on G/H = Z/p
        out[ci + m * np.arange(p)] = w @ G[:, ci]   # b = c + m j  <->  chi_psi * eta_j
    return out

def bound(n):
    s, k = 0, n
    while k > 1:
        p = smallest_prime(k); s += p + 2; k //= p
    return n * s

rng = np.random.default_rng(0)
for n in [2 ** 10, 3 ** 6, 1000, 997]:
    f = rng.standard_normal(n) + 1j * rng.standard_normal(n)
    ops = 0
    t0 = time.perf_counter(); F = fft(f); t1 = time.perf_counter()
    err = np.max(np.abs(F - np.fft.fft(f) / n))
    print(f"n = {n:5d}: error vs numpy {err:.1e}, operations {ops:8d}, bound {bound(n):8d}, n^2 = {n*n:8d}, {1e3*(t1-t0):.1f} ms")

def walsh_hadamard(F):
    """In place: returns sum_x F(x) (-1)^{S.x} for every S, in n 2^n additions."""
    F = F.astype(float).copy()
    h = 1
    while h < len(F):
        for i in range(0, len(F), 2 * h):
            a, b = F[i:i + h].copy(), F[i + h:i + 2 * h].copy()
            F[i:i + h], F[i + h:i + 2 * h] = a + b, a - b
        h *= 2
    return F

n = 4
F = rng.choice([-1, 1], 2 ** n)
H = np.array([[(-1) ** bin(S & x).count('1') for x in range(2 ** n)] for S in range(2 ** n)])
print("Walsh-Hadamard agrees with the definition:", np.allclose(walsh_hadamard(F), H @ F))

The recursive transform agrees with `numpy` to rounding error in all four cases. For $n=2^{10}$ it uses about $3\cdot10^4$ operations against $10^6$ for the definition, comfortably under the bound $n\sum(p_i+2)=40\,960$ of Theorem 7.8; $3^6$ and $1000=2^3\cdot5^3$ are similar. For the prime $997$ there is nothing to split, and the count is the full $997^2\approx10^6$: this is where Rader's trick (convolution over the cyclic group $(\mathbb Z/997)^*$ of order $996=4\cdot3\cdot83$) or Bluestein's chirp trick takes over in real libraries. The Walsh–Hadamard transform is the same lemma for $\mathbb F_2^n$, with $H$ a coordinate hyperplane at every level: each pass does $2^{n-1}$ butterflies $(a,b)\mapsto(a+b,a-b)$ and no multiplication at all, which is why it is used in quantum computing (the Hadamard gate), in coding theory and in fast image transforms.

### P7.2

In [ ]:
def all_minors(n):
    W = np.exp(2j * np.pi * np.outer(np.arange(n), np.arange(n)) / n)
    dets = []
    for k in range(1, n + 1):
        for R in itertools.combinations(range(n), k):
            for C in itertools.combinations(range(n), k):
                dets.append(abs(np.linalg.det(W[np.ix_(R, C)])))
    return np.array(dets)

for n in [7, 8]:
    d = all_minors(n)
    print(f"n = {n}: {len(d)} minors, smallest |det| = {d.min():.3e}, singular (|det| < 1e-9): {(d < 1e-9).sum()}")

def recover(p, A, freqs, values, k):
    """All k-sparse f with hat f = values on freqs (brute force over supports)."""
    sols = []
    for S in itertools.combinations(range(p), k):
        M = np.exp(-2j * np.pi * np.outer(freqs, S) / p) / p
        c, *_ = np.linalg.lstsq(M, values, rcond=None)
        if np.linalg.norm(M @ c - values) < 1e-9 and np.all(np.abs(c) > 1e-9):
            sols.append((S, np.round(c, 6)))
    return sols

p, k = 31, 3
S0 = sorted(rng.choice(p, k, replace=False))
f = np.zeros(p, complex); f[S0] = rng.standard_normal(k) + 1j * rng.standard_normal(k)
freqs = np.sort(rng.choice(p, 2 * k, replace=False))
values = np.fft.fft(f)[freqs] / p
sols = recover(p, S0, freqs, values, k)
print("true support", S0, "| frequencies kept", freqs, "| supports found:", [s for s, _ in sols])

n = 32                                   # a signal on the subgroup 8Z/32Z, spectrum on 4Z/32Z
f = np.zeros(n); f[::8] = 1
F = np.fft.fft(f) / n
print("Z/32: |supp f| =", np.count_nonzero(f), " |supp hat f| =", np.count_nonzero(np.abs(F) > 1e-12), " sum =", 4 + np.count_nonzero(np.abs(F) > 1e-12), "< n+1 = 33")

For $p=7$ all $3431$ minors are non-zero; the smallest has absolute value about $0.87$, comfortably away from zero, so Chebotarev's lemma is visible even in floating point. For $n=8$, $1396$ of the $12\,869$ minors vanish, for instance whenever the rows and the columns are cosets of a subgroup, as in Exercise 7.3. In the recovery experiment, among all $\binom{31}3=4495$ supports exactly one is consistent with the six kept coefficients, and it is the true one, as Theorem 7.11 guarantees, whichever six frequencies were drawn. On $\mathbb Z/32\mathbb Z$ the indicator of the subgroup $\{0,8,16,24\}$ has $4$ non-zero values and $8$ non-zero Fourier coefficients: $12$ is far below $33$. Brute force over supports is exponential in $k$; compressed sensing replaces it by $\ell^1$ minimisation, a linear program, which provably finds the same answer when the frequencies are random and slightly more numerous (of order $k\log p$).

### P7.3

In [ ]:
def is_cap(points, n):
    """Fourier count of Lemma 7.15: T(A) = N^2 sum hat(1_A)^3 must equal |A|."""
    A = np.zeros((3,) * n)
    for x in points:
        A[tuple(x)] = 1
    N = 3 ** n
    hat = np.fft.fftn(A) / N                 # hat 1_A(xi) = E_x 1_A(x) omega^{-xi.x}
    T = N ** 2 * np.sum(hat ** 3).real
    return round(T) == len(points)

def greedy_cap(n, runs, rng):
    best = []
    pts = np.array(list(itertools.product(range(3), repeat=n)))
    for _ in range(runs):
        order = rng.permutation(len(pts))
        cap, forbidden = [], set()
        for i in order:
            x = tuple(pts[i])
            if x in forbidden:
                continue
            for y in cap:                    # x and y forbid the third point of their line
                forbidden.add(tuple((-np.array(x) - np.array(y)) % 3))
            cap.append(x)
        if len(cap) > len(best):
            best = cap
    return best

true_max = {1: 2, 2: 4, 3: 9, 4: 20, 5: 45, 6: 112}
print(" n  greedy  max  Meshulam 2*3^n/n  2.756^n   Fourier check")
for n in range(1, 7):
    cap = greedy_cap(n, 400 if n < 6 else 60, rng)
    print(f"{n:2d}  {len(cap):6d}  {true_max[n]:4d}  {2 * 3 ** n / n:14.1f}  {2.756 ** n:8.1f}   {is_cap(cap, n)}")

cards = np.array(list(itertools.product(range(3), repeat=4)))
trials, noset = 20000, 0
for _ in range(trials):
    deal = cards[rng.choice(81, 12, replace=False)]
    s = set(map(tuple, deal))
    found = any(tuple((-deal[i] - deal[j]) % 3) in s for i in range(12) for j in range(i + 1, 12))
    noset += not found
print(f"12 random cards contain no set in {100 * noset / trials:.2f}% of deals")

The greedy caps reach the true maximum in dimensions $1$ to $4$ and fall behind afterwards ($38$ instead of $45$ in dimension $5$, $76$ instead of $112$ in dimension $6$): finding maximal caps is hard, and the values $45$ (Edel, Ferret, Landjev and Storme, 2002) and $112$ (Potechin, 2008) needed clever constructions and computer-assisted proofs. In these small dimensions Meshulam's bound is the better one; the polynomial bound $3\cdot2.756^n$ of Exercise 7.7 only overtakes $\frac{2\cdot3^n}n$ from $n=52$ on, but then it wins by an exponential factor. The Fourier check confirms that each greedy set has exactly $|A|$ solutions of $x+y+z=0$, the trivial ones. Finally, about $3\%$ of random twelve-card deals have no set, which matches the experience of SET players that the dealer rarely needs to add cards.

### P7.4

In [ ]:
n = 10
N = 2 ** n
xs = np.arange(N)
bits = (xs[:, None] >> np.arange(n)) & 1
S = 0b1011001101
lin = (-1.0) ** (bin_dot := (bits @ np.array([(S >> i) & 1 for i in range(n)])) % 2)

def flip(F, frac):
    G = F.copy(); idx = rng.choice(N, int(frac * N), replace=False); G[idx] *= -1; return G

maj = np.sign((1 - 2 * bits[:, 0]) + (1 - 2 * bits[:, 1]) + (1 - 2 * bits[:, 2]))
cases = {"linear": lin, "5% flipped": flip(lin, 0.05), "20% flipped": flip(lin, 0.20),
         "majority of 3 bits": maj, "random": rng.choice([-1.0, 1.0], N)}
print(f"{'function':20s} {'simulated':>10s} {'exact':>8s} {'dist to linear':>15s}")
for name, F in cases.items():
    x = rng.integers(0, N, 200000); y = rng.integers(0, N, 200000)
    sim = np.mean(F[x] * F[y] == F[x ^ y])
    hat = walsh_hadamard(F) / N
    exact = 0.5 + 0.5 * np.sum(hat ** 3)
    print(f"{name:20s} {sim:10.4f} {exact:8.4f} {(1 - hat.max()) / 2:15.4f}")

Simulation and formula agree to about three decimals. A linear function always passes. Flipping a fraction $\varepsilon=5\%$ of the values gives an acceptance probability near $0.865$: the test rejects when one or three of the queried points are corrupted, which happens with probability $3\varepsilon(1-\varepsilon)^2+\varepsilon^3=0.135$ if the three points were independent (they almost are). With $20\%$ flipped the probability drops to about $0.61$, and the distance to the nearest linear function is still exactly $0.2$, since the flips are too few to bring another character closer. The majority function passes with probability $\frac58$, as in Exercise 7.6, while its distance to linear is $\frac14$. A random function passes about half the time, and Theorem 7.17 explains why the random function is far from every character: its largest Fourier coefficient is of order $\sqrt{n/2^n}$. The test is the prototype of *property testing*: a constant number of queries decides, with high confidence, whether an object of size $2^n$ is close to having a property.